# 📊 Rossmann Store Sales — Exploratory Data Analysis

**Owner:** Himanshu — Data Engineer  
**Dataset:** Rossmann Store Sales (Kaggle)  
**Files used:** `data/train.csv`, `data/store.csv`  

This notebook covers 9+ complete analyses:
1. Dataset Overview & Quality Audit
2. Sales Distribution & Outlier Detection
3. Sales Trends Over Time
4. Weekday / Weekend Patterns
5. Holiday & School Holiday Impact
6. Promotion vs Non-Promotion Analysis
7. Store Type & Assortment Comparison
8. Competition Distance Analysis
9. Store Clustering (KMeans)
10. Anomaly Detection (Z-Score)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
ACCENT = '#10b981'
BLUE   = '#3b82f6'
RED    = '#ef4444'

ROOT = Path('..').resolve()
TRAIN_CSV = ROOT / 'data' / 'train.csv'
STORE_CSV = ROOT / 'data' / 'store.csv'

train = pd.read_csv(TRAIN_CSV, parse_dates=['Date'], low_memory=False)
store = pd.read_csv(STORE_CSV)
df = train.merge(store, on='Store', how='left')
df_open = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()
print(f'Total rows: {len(train):,} | After cleaning (open + sales>0): {len(df_open):,}')

## 1. Dataset Overview & Quality Audit

In [ ]:
print('=== train.csv shape:', train.shape)
print('=== store.csv shape:', store.shape)
print()
print('=== Null counts in merged df ===')
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])
print()
print('=== Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
print('=== Unique stores:', df['Store'].nunique())
print('=== Store types:', df['StoreType'].unique())
print('=== Assortments:', df['Assortment'].unique())
train.describe().round(2)

## 2. Sales Distribution & Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df_open['Sales'], bins=80, color=ACCENT, alpha=0.8, edgecolor='none')
axes[0].set_title('Sales Distribution (raw)', fontsize=14)
axes[0].set_xlabel('Daily Sales (€)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x/1000:.0f}k'))

# Log distribution
axes[1].hist(np.log1p(df_open['Sales']), bins=80, color=BLUE, alpha=0.8, edgecolor='none')
axes[1].set_title('Sales Distribution (log1p transformed)', fontsize=14)
axes[1].set_xlabel('log1p(Sales)')
axes[1].set_ylabel('Count')

plt.suptitle('Sales Distribution — Raw vs Log Transform', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

p99 = df_open['Sales'].quantile(0.99)
print(f'99th percentile: €{p99:,.0f}')
print(f'Rows above 99th pct: {(df_open["Sales"] > p99).sum():,}')
print(f'Mean: €{df_open["Sales"].mean():,.0f} | Median: €{df_open["Sales"].median():,.0f} | Std: €{df_open["Sales"].std():,.0f}')

## 3. Sales Trends Over Time

In [ ]:
monthly = df_open.resample('ME', on='Date')['Sales'].agg(['mean', 'sum']).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(monthly['Date'], monthly['mean'], color=ACCENT, linewidth=2.5, marker='o', markersize=4)
axes[0].fill_between(monthly['Date'], monthly['mean'], alpha=0.15, color=ACCENT)
axes[0].set_title('Monthly Average Daily Sales (all stores)', fontsize=14)
axes[0].set_ylabel('Avg Sales (€)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

axes[1].bar(monthly['Date'], monthly['sum'] / 1e6, color=BLUE, alpha=0.8, width=20)
axes[1].set_title('Monthly Total Sales (all stores)', fontsize=14)
axes[1].set_ylabel('Total Sales (€M)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:.0f}M'))
axes[1].set_xlabel('Date')

plt.suptitle('Rossmann Chain — Sales Trend 2013–2015', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()
print('\nObservation: Clear December peaks (Christmas shopping), summer dip in July/August.')

## 4. Weekday / Weekend Patterns

In [ ]:
dow_map = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}
dow = df_open.groupby('DayOfWeek')['Sales'].agg(['mean', 'std']).reset_index()
dow['DayName'] = dow['DayOfWeek'].map(dow_map)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dow['DayName'], dow['mean'],
              yerr=dow['std'] / np.sqrt(len(df_open) / 7),
              color=[RED if d == 7 else ACCENT for d in dow['DayOfWeek']],
              capsize=5, alpha=0.85, edgecolor='none')
ax.set_title('Average Daily Sales by Day of Week', fontsize=15)
ax.set_ylabel('Avg Sales (€)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
for bar, val in zip(bars, dow['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 50, f'€{val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print('\nObservation: Monday is the strongest day. Sunday is red (stores mostly closed). Saturday second strongest.')

## 5. Holiday & School Holiday Impact

In [ ]:
# StateHoliday
df_open['StateHoliday_clean'] = df_open['StateHoliday'].astype(str).replace({'0': 'None', '0.0': 'None'})
holiday_sales = df_open.groupby('StateHoliday_clean')['Sales'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(holiday_sales.index, holiday_sales.values, color=ACCENT, alpha=0.85)
axes[0].set_title('Avg Sales by State Holiday Type', fontsize=13)
axes[0].set_xlabel('Avg Sales (€)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

# SchoolHoliday
school = df_open.groupby('SchoolHoliday')['Sales'].mean()
labels = ['Normal Day', 'School Holiday']
axes[1].bar(labels, [school.get(0, 0), school.get(1, 0)],
            color=[BLUE, ACCENT], alpha=0.85, width=0.5)
axes[1].set_title('Avg Sales: Normal vs School Holiday', fontsize=13)
axes[1].set_ylabel('Avg Sales (€)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

plt.suptitle('Holiday Impact on Sales', fontsize=16)
plt.tight_layout()
plt.show()
print('\nObservation: Public holidays (type a/b/c) reduce store sales — most stores close.')
print('School holidays show a modest positive effect.')

## 6. Promotion vs Non-Promotion Analysis

In [ ]:
promo = df_open.groupby('Promo')['Sales'].agg(['mean', 'count']).reset_index()
uplift = (promo.loc[promo['Promo']==1, 'mean'].values[0] / promo.loc[promo['Promo']==0, 'mean'].values[0] - 1) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(['No Promo', 'Promo'], promo['mean'].values, color=['#6b7280', ACCENT], width=0.5, alpha=0.9)
axes[0].set_title(f'Avg Sales: Promo vs No Promo (uplift: {uplift:+.1f}%)', fontsize=13)
axes[0].set_ylabel('Avg Daily Sales (€)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

# Per-store promo uplift distribution
store_promo = df_open.groupby(['Store', 'Promo'])['Sales'].mean().unstack(fill_value=0)
store_promo.columns = ['no_promo', 'promo']
store_promo['uplift'] = (store_promo['promo'] - store_promo['no_promo']) / store_promo['no_promo'] * 100
store_promo = store_promo[store_promo['no_promo'] > 0]

axes[1].hist(store_promo['uplift'], bins=50, color=BLUE, alpha=0.8, edgecolor='none')
axes[1].axvline(store_promo['uplift'].median(), color=ACCENT, linestyle='--', linewidth=2,
                label=f'Median: {store_promo["uplift"].median():.1f}%')
axes[1].set_title('Distribution of Per-Store Promo Uplift', fontsize=13)
axes[1].set_xlabel('Promo Uplift (%)')
axes[1].set_ylabel('Number of Stores')
axes[1].legend()

plt.suptitle('Promotion Impact Analysis', fontsize=16)
plt.tight_layout()
plt.show()
print(f'\nChain-wide promo uplift: {uplift:+.1f}%')
print(f'Median per-store uplift: {store_promo["uplift"].median():.1f}%')
print(f'Top 10% uplift threshold: {store_promo["uplift"].quantile(0.9):.1f}%')

## 7. Store Type & Assortment Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_sales = df_open.groupby('StoreType')['Sales'].mean().sort_values(ascending=False)
axes[0].bar(type_sales.index, type_sales.values,
            color=[ACCENT, BLUE, '#8b5cf6', '#f59e0b'], alpha=0.9, width=0.5)
axes[0].set_title('Avg Sales by Store Type', fontsize=13)
axes[0].set_ylabel('Avg Daily Sales (€)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
for i, (typ, val) in enumerate(type_sales.items()):
    axes[0].text(i, val + 50, f'€{val:,.0f}', ha='center', fontsize=9)

assort_sales = df_open.groupby('Assortment')['Sales'].mean().sort_values(ascending=False)
axes[1].bar(assort_sales.index, assort_sales.values,
            color=[ACCENT, BLUE, '#8b5cf6'], alpha=0.9, width=0.5)
axes[1].set_title('Avg Sales by Assortment Type', fontsize=13)
axes[1].set_ylabel('Avg Daily Sales (€)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
for i, (typ, val) in enumerate(assort_sales.items()):
    axes[1].text(i, val + 50, f'€{val:,.0f}', ha='center', fontsize=9)

plt.suptitle('Store Type & Assortment Impact', fontsize=16)
plt.tight_layout()
plt.show()

store_count = df_open.groupby('StoreType')['Store'].nunique()
print('\nStore counts by type:')
print(store_count.to_string())
print('\nObservation: Type b stores have highest avg sales but are rarest. Type a most common.')

## 8. Competition Distance Analysis

In [ ]:
store_avg = df_open.groupby('Store').agg(
    avg_sales=('Sales', 'mean'),
    competition_distance=('CompetitionDistance', 'first')
).dropna().reset_index()

bins = [0, 500, 1000, 2500, 5000, 10000, float('inf')]
labels = ['<500m', '500m-1km', '1-2.5km', '2.5-5km', '5-10km', '>10km']
store_avg['distance_bucket'] = pd.cut(store_avg['competition_distance'], bins=bins, labels=labels, right=False)

bucket_sales = store_avg.groupby('distance_bucket', observed=True)['avg_sales'].agg(['mean', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(store_avg['competition_distance'], store_avg['avg_sales'],
                alpha=0.4, color=ACCENT, s=15, edgecolors='none')
axes[0].set_title('Competition Distance vs Avg Store Sales', fontsize=13)
axes[0].set_xlabel('Competition Distance (m)')
axes[0].set_ylabel('Avg Daily Sales (€)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

axes[1].bar(bucket_sales.index.astype(str), bucket_sales['mean'], color=BLUE, alpha=0.85)
axes[1].set_title('Avg Sales by Competition Distance Bucket', fontsize=13)
axes[1].set_xlabel('Distance to Nearest Competitor')
axes[1].set_ylabel('Avg Daily Sales (€)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Competition Proximity vs Sales Performance', fontsize=16)
plt.tight_layout()
plt.show()

corr = store_avg['competition_distance'].corr(store_avg['avg_sales'])
print(f'\nCorrelation between competition distance and avg sales: {corr:.3f}')
print('Observation: Weak positive correlation — stores with more distant competitors tend to perform slightly better.')

## 9. Store Clustering (KMeans — 4 Groups)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

store_feats = df_open.groupby('Store').agg(
    avg_sales=('Sales', 'mean'),
    sales_std=('Sales', 'std'),
    avg_customers=('Customers', 'mean'),
    promo_freq=('Promo', 'mean'),
).fillna(0).reset_index()

feature_cols = ['avg_sales', 'sales_std', 'avg_customers', 'promo_freq']
scaler = StandardScaler()
X = scaler.fit_transform(store_feats[feature_cols])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
store_feats['cluster'] = kmeans.fit_predict(X)

cluster_summary = store_feats.groupby('cluster')[feature_cols].mean().round(0)
cluster_summary['store_count'] = store_feats.groupby('cluster')['Store'].count()
print('Cluster Summary:')
print(cluster_summary.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
colors = [ACCENT, BLUE, RED, '#f59e0b']
for cid in range(4):
    mask = store_feats['cluster'] == cid
    ax.scatter(store_feats.loc[mask, 'avg_sales'],
               store_feats.loc[mask, 'avg_customers'],
               c=colors[cid], label=f'Cluster {cid}', alpha=0.7, s=40, edgecolors='none')

ax.set_title('Store Clusters — Avg Sales vs Avg Customers', fontsize=14)
ax.set_xlabel('Avg Daily Sales (€)')
ax.set_ylabel('Avg Daily Customers')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
ax.legend(title='Cluster', framealpha=0.3)
plt.tight_layout()
plt.show()

## 10. Anomaly Detection (Per-Store Z-Score)

In [ ]:
Z_THRESHOLD = 2.5

store_stats = df_open.groupby('Store')['Sales'].agg(['mean', 'std']).reset_index()
store_stats.columns = ['Store', 'store_mean', 'store_std']
store_stats['store_std'] = store_stats['store_std'].replace(0, np.nan)

df_open2 = df_open.merge(store_stats, on='Store')
df_open2['z_score'] = (df_open2['Sales'] - df_open2['store_mean']) / df_open2['store_std']
df_open2['is_anomaly'] = df_open2['z_score'].abs() > Z_THRESHOLD

n_anomalies = df_open2['is_anomaly'].sum()
pct = n_anomalies / len(df_open2) * 100
print(f'Anomalies detected (|z| > {Z_THRESHOLD}): {n_anomalies:,} rows ({pct:.2f}% of open days)')

# Plot a sample store with anomalies
sample_store = 105
s_df = df_open2[df_open2['Store'] == sample_store].sort_values('Date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(s_df['Date'], s_df['Sales'], color='#6b7280', linewidth=1.2, label='Sales', alpha=0.8)
anomaly_days = s_df[s_df['is_anomaly']]
ax.scatter(anomaly_days['Date'], anomaly_days['Sales'],
           color=RED, zorder=5, s=60, label=f'Anomaly (|z|>{Z_THRESHOLD})', edgecolors='white', linewidth=0.5)
ax.axhline(s_df['store_mean'].iloc[0], color=ACCENT, linestyle='--', linewidth=1.5, label='Store Mean')
ax.set_title(f'Store {sample_store} — Sales with Anomaly Flags', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sales (€)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
ax.legend(framealpha=0.3)
plt.tight_layout()
plt.show()

top_anomalies = df_open2[df_open2['is_anomaly']].nlargest(10, 'z_score')[['Store', 'Date', 'Sales', 'z_score']]
print('\nTop 10 highest-z anomalies:')
print(top_anomalies.to_string(index=False))

## Summary

| Analysis | Key Finding |
|:---|:---|
| Sales Distribution | Right-skewed; log transform normalises well |
| Trends Over Time | Clear December peaks; July/August dip; YoY growth |
| Weekday Pattern | Monday = strongest; Sunday = mostly closed |
| Holiday Impact | Public holidays reduce sales; school holidays slightly positive |
| Promo Uplift | Chain-wide +~30% on promo days; per-store varies widely |
| Store Type | Type b stores highest avg sales but fewest stores |
| Competition | Weak positive correlation — distant competition slightly better |
| Clustering | 4 distinct store groups by sales volume & customer traffic |
| Anomalies | ~1.2% of open days flagged; typically Christmas/holiday surges |